# Module 1 · Trustworthy data and leakage-free pipelines
    **Outcomes:** (a) choose the right approach, (b) detect leakage and build a leakage-free pipeline with a fitting validation strategy.

    The case study for the whole day: a **loan-default model**. Each row is a loan application;
    the target `defaulted_in_next_30_days` is 1 when the borrower defaulted (about 9% of loans).

> **How to use this notebook.** Run cells top to bottom with **Shift + Enter**.
> Cells marked **✏️ TODO** are yours to complete. **🏁 Checkpoint** cells tell you when a module is done.
> Stuck? Ask your neighbour first, then the trainer. Solutions are released after the session.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, recall_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from course_utils import check_leak_verdicts, check_pipeline, make_loans

TARGET = "defaulted_in_next_30_days"
df = make_loans()
print(df.shape, f"· default rate {df[TARGET].mean():.1%}")
df.head()

## A · Data quality first
Five dimensions: **integrity, completeness, accuracy, timeliness, validity**. Check them before you trust any metric.

In [ ]:
num = df.select_dtypes("number")
q1, q3 = num.quantile(0.25), num.quantile(0.75)
outside = ((num < q1 - 1.5 * (q3 - q1)) | (num > q3 + 1.5 * (q3 - q1))).mean() * 100

report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_%": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
    "outside_IQR_fences_%": outside.round(1),
})
report

Is the missing income **random**? Compare applicants with short and long employment:

In [ ]:
short = df["employment_years"] < 2
print(f"income missing · employment < 2 years : {df.loc[short, 'annual_income'].isna().mean():.0%}")
print(f"income missing · employment ≥ 2 years : {df.loc[~short, 'annual_income'].isna().mean():.0%}")

A big gap means the blanks depend on another column: **MAR**. Impute using that information (inside the pipeline),
and consider a `was_missing` flag.

## B · Hands-on: find every leak (15 min)
For each column ask one question: **would I know this value at the moment the loan is approved?**

| Verdict | Meaning |
|---|---|
| `LEAK` | Created after, or because of, the outcome. Never use it. |
| `RISKY` | Usable only with care (as a split key, or if measured at the right time, or imputed correctly). |
| `SAFE` | Known on the application form. |

8 minutes alone, 3 minutes with your neighbour, then the reveal.

In [ ]:
# ✏️ TODO: replace each "?" with "LEAK", "RISKY" or "SAFE", then run the cell
verdicts = {
    "customer_id": "?",
    "loan_approval_date": "?",
    "loan_amount": "?",
    "annual_income": "?",
    "employment_years": "?",
    "credit_score": "?",
    "days_since_last_payment": "?",
    "existing_loans": "?",
    "loan_type": "?",
    "region": "?",
    "days_to_default": "?",
    "post_approval_status_flag": "?",
}
check_leak_verdicts(verdicts)

## C · What leakage does to a score
Here is a model built the wrong way: it uses the two leaked columns and fills the gaps **before** splitting.

In [ ]:
leaky_cols = ["loan_amount", "credit_score", "days_to_default", "post_approval_status_flag"]
X_leak = pd.get_dummies(df[leaky_cols], columns=["post_approval_status_flag"]).fillna(-1)
X_tr, X_te, y_tr, y_te = train_test_split(X_leak, df[TARGET], test_size=0.2, random_state=42, stratify=df[TARGET])
leaky = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
print(f"Leaky model ROC-AUC: {roc_auc_score(y_te, leaky.predict_proba(X_te)[:, 1]):.3f}   ← too good to be true")

always_no = np.zeros(len(y_te), dtype=int)
print(f"'Always predict no default': accuracy {(always_no == y_te).mean():.1%}, recall {recall_score(y_te, always_no):.0%}")

Two lessons from the ice breaker: a perfect score usually means a leak, and **accuracy hides a model that catches nobody**
when defaults are rare. From now on we report **ROC-AUC and recall**.

## D · Hands-on: the leakage-free pipeline (20 min)
The safe order: **split by time first → fit every step inside a `Pipeline` on training rows only → score the newest 20% once.**

In [ ]:
FEATURES_NUM = ["loan_amount", "annual_income", "employment_years", "credit_score",
                "days_since_last_payment", "existing_loans"]
FEATURES_CAT = ["loan_type", "region"]

# 1. ✏️ TODO: sort by loan_approval_date; the oldest 80% trains, the newest 20% tests
df_sorted = ...
cut = ...
train, test = ..., ...

X_train, X_test = train[FEATURES_NUM + FEATURES_CAT], test[FEATURES_NUM + FEATURES_CAT]
y_train, y_test = train[TARGET], test[TARGET]

# 2. ✏️ TODO: numeric = median imputer then StandardScaler
#             categorical = most_frequent imputer then OneHotEncoder(handle_unknown="ignore")
num_pipe = Pipeline([...])
cat_pipe = Pipeline([...])
pre = ColumnTransformer([("num", num_pipe, FEATURES_NUM), ("cat", cat_pipe, FEATURES_CAT)])

# 3. ✏️ TODO: chain `pre` with LogisticRegression(max_iter=1000, class_weight="balanced"), fit on TRAIN only
model = Pipeline([...])
model.fit(...)

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
pred = model.predict(X_test)
print(f"Newest 20% · ROC-AUC {roc_auc_score(y_test, proba):.3f} · recall {recall_score(y_test, pred):.0%}")
print(classification_report(y_test, pred, digits=3, zero_division=0))

Lower than the leaky model, and **honest**: this is what next month's applicants will see.

### 🏁 Checkpoint · Module 1

In [ ]:
check_pipeline(model, X_train, X_test, train["loan_approval_date"], test["loan_approval_date"])

## E · Validate the way the model will be used
While tuning, use **TimeSeriesSplit** on the training rows: always train on the past, score the next slice.

In [ ]:
scores = cross_val_score(model, X_train, y_train, cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc")
print("ROC-AUC per fold:", scores.round(3), "· mean", scores.mean().round(3))

**Stretch:** customers repeat in this data (`customer_id`). How would you keep each customer on one side of the split?
(Hint: `GroupKFold` or `GroupShuffleSplit` with `groups=df["customer_id"]`.)

### Quick quiz
1. Why is fitting a scaler on the full dataset a problem?
2. When should you prefer a time-based split?
3. Which columns here are pure target leakage?